# Анализ ДТП в Великобритании

Групповой проект по дисциплине «Архитектура больших данных». Источник — открытый набор STATS19 Collisions. Одна запись соответствует зарегистрированному ДТП с пострадавшими.

**Порядок запуска.** Поместите этот ноутбук в корень `road-safety-big-data` рядом с `.env`, `compose.yaml` и `etl.py`. Запустите Docker Desktop, затем в терминале проекта выполните `docker compose up -d postgres` и `python etl.py`. Дождитесь сообщения о завершении ETL. Выберите Python из вашего виртуального окружения, нажмите «Перезапустить», затем «Выполнить все». Исходный CSV должен находиться в `data/raw/dft-road-casualty-statistics-collision-1979-latest-published-year.csv`.

Ноутбук получает записи из PostgreSQL и создаёт таблицы результатов, интерактивные HTML-графики и PNG-изображения в `results/`. Установка недостающих Python-пакетов выполняется первой ячейкой. Для неё нужен интернет. Сохранённых результатов прежнего запуска здесь нет. Все числа и выводы рассчитываются на вашей базе.

Superset запускается отдельно. В последнем разделе приведены действия для подключения таблицы и создания дашборда. Выполнение ноутбука само по себе не создаёт дашборд в Superset.


## 1. Архитектура и ответственность участников

| Участник | Ответственность |
|---|---|
| Иван Ф | PostgreSQL, Docker, параметры подключения и окружение |
| Артём | ETL, очистка и загрузка полного набора, контроль качества |
| Иван А | Статистические проверки, регрессионный анализ и классификация |
| Кира | Формулировки гипотез, интерактивные графики, Superset и выводы |

Данные проходят последовательность `CSV → ETL на Python → PostgreSQL → SQL и Python → графики и Superset`.

В PostgreSQL сохраняются необходимые исследованию восемь столбцов за все годы. Годовая статистика строится по всей загруженной таблице. Для проверки гипотез и моделирования выбран период 2022–2024 годов, чтобы уменьшить смешивание сильно отличающихся исторических условий. Применимость выводов ограничена этим периодом. Перечень полей можно расширить отдельной миграцией и обновлением ETL.

PostgreSQL обеспечивает типизированное хранение, уникальность идентификаторов и SQL-агрегации. Чтение CSV порциями ограничивает расход памяти при загрузке. Docker задаёт окружение PostgreSQL. Superset использует подключение к базе для интерактивных запросов.


## 2. PostgreSQL и Docker

В проекте используется сервис `postgres` с образом `postgres:17`. При первой инициализации переменные `POSTGRES_DB`, `POSTGRES_USER`, `POSTGRES_PASSWORD` создают базу и пользователя. В `.env` также задан `POSTGRES_PORT`. Пароль хранится локально, файл `.env` включён в `.gitignore`.

Именованный volume `postgres_data` подключён к `/var/lib/postgresql/data`. Он сохраняет базу между обычными перезапусками контейнера. Файл CSV хранится на Mac, Python передаёт обработанные данные через подключение к PostgreSQL.

В терминале корня проекта:

```bash
docker compose up -d postgres
docker compose ps
docker compose exec postgres psql -U road_user -d road_safety -c "SELECT current_database(), current_user;"
```

Статус `healthy` показывает готовность сервера. Если таблица ещё не создана, выполните:

```bash
docker compose exec -T postgres psql -U road_user -d road_safety -v ON_ERROR_STOP=1 < sql/schema.sql
```

Таблица `public.collisions` содержит `collision_index TEXT PRIMARY KEY`, `collision_year SMALLINT NOT NULL`, `collision_severity`, `speed_limit`, `light_conditions`, `police_force` типа `SMALLINT`, `date DATE` и `time TIME`. Индекс по `collision_year` помогает отбирать период.

Python на Mac использует адрес `127.0.0.1`. Superset в отдельном Docker Compose на Mac подключается через `host.docker.internal` и опубликованный порт. Если оба сервиса находятся в одной сети Compose, используется имя сервиса `postgres` и внутренний порт `5432`.


## 3. ETL и заполнение таблицы

**Extract.** Скрипт `etl.py` читает весь исходный CSV блоками по 100 000 строк, выбирая восемь используемых полей. Размер исходного файла проверяется отдельно. Маленький просмотр данных не заменяет полную загрузку.

**Transform.** Идентификаторы сохраняются как строки, удаляются окружающие пробелы. Числовые поля проверяются на целочисленность и диапазон `SMALLINT`. Даты разбираются как день/месяц/год, время — как часы:минуты. Дата, не соответствующая году записи, становится пропуском. Строки с некорректным идентификатором или годом записываются в файл отклонений. Некорректные необязательные значения становятся `NULL`. Отрицательные коды источника могут сохраняться как коды неизвестного значения и учитываются при фильтрации конкретных расчётов.

**Load.** Обработанный блок передаётся через `COPY` во временную таблицу и затем в `public.collisions`. Совпадающие записи с одним ID пропускаются. Разные записи с одним ID останавливают загрузку и требуют разбора. Блоки фиксируются отдельными транзакциями. Повторный запуск может продолжить загрузку с пропуском совпадающих ранее сохранённых записей.

```bash
python etl.py
```

Дождитесь завершения загрузки перед анализом. Отчёт `results/etl_report_*.json` содержит количество прочитанных, добавленных, повторных и отклонённых записей, пропуски и итоговый статус. Файлы `etl_rejects_*.csv` появляются при наличии отклонений. Данный ноутбук проверяет отчёт и базу, но не запускает повторный ETL автоматически.


## 4. Подготовка Python

Первая ячейка устанавливает только отсутствующие зависимости в выбранное ядро. Затем сохраняется список фактически использованных версий. Если окружение проекта уже активировано, выберите его ядро в правом верхнем углу VS Code.


In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    'pandas': 'pandas', 'numpy': 'numpy', 'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn', 'scipy': 'scipy', 'sqlalchemy': 'sqlalchemy',
    'psycopg': 'psycopg[binary]', 'dotenv': 'python-dotenv',
    'plotly': 'plotly', 'nbformat': 'nbformat',
}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print('Установка:', ', '.join(missing))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Python:', sys.executable)


In [ ]:
import json
import os
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown
from scipy.stats import norm, ttest_1samp
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
)

candidates = [Path.cwd(), Path.cwd() / 'road-safety-big-data']
ROOT = next((p for p in candidates if (p / '.env').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Откройте ноутбук в папке проекта рядом с .env.')
load_dotenv(ROOT / '.env', override=True)
RESULTS = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
START_YEAR, END_YEAR = 2022, 2024
ALPHA = 0.05
RANDOM_STATE = 42
pio.renderers.default = 'notebook_connected'

versions = {p: version(p) for p in [
    'pandas', 'numpy', 'matplotlib', 'scikit-learn', 'scipy',
    'sqlalchemy', 'psycopg', 'python-dotenv', 'plotly', 'nbformat',
]}
(RESULTS / 'analysis_environment.json').write_text(
    json.dumps({'python': sys.version, 'packages': versions}, indent=2),
    encoding='utf-8',
)
print('Папка проекта:', ROOT)
print('Период исследования:', START_YEAR, '–', END_YEAR)


## 5. Подключение и контроль завершения ETL

Пароль берётся из `.env` и не выводится. При наличии отчёта ETL проверяется статус последнего запуска и согласованность количеств. Если загрузка ещё выполняется, сначала дождитесь её окончания. При отсутствии отчёта проверяется непустая база, но завершённость исходной загрузки не считается доказанной.


In [ ]:
required_env = ['POSTGRES_DB', 'POSTGRES_USER', 'POSTGRES_PASSWORD']
absent = [name for name in required_env if not os.getenv(name)]
if absent:
    raise ValueError('Заполните .env: ' + ', '.join(absent))

reports = sorted(RESULTS.glob('etl_report_*.json'))
etl_report = None
if reports:
    etl_report = json.loads(reports[-1].read_text(encoding='utf-8'))
    if etl_report.get('status') != 'completed':
        raise RuntimeError('Последний ETL не завершён успешно. Проверьте ' + reports[-1].name)
    display(pd.DataFrame({
        'Показатель': ['Прочитано', 'Добавлено', 'Повторов', 'Отклонено'],
        'Значение': [etl_report[k] for k in [
            'rows_read', 'rows_inserted', 'rows_duplicate', 'rows_rejected']],
    }))
    counted = sum(etl_report[k] for k in [
        'rows_inserted', 'rows_duplicate', 'rows_rejected'])
    if counted != etl_report['rows_read']:
        raise ValueError('Нарушен баланс строк в отчёте ETL.')
else:
    print('Отчёт ETL отсутствует. Полноту загрузки нужно подтвердить отдельно.')

url = URL.create(
    'postgresql+psycopg', username=os.environ['POSTGRES_USER'],
    password=os.environ['POSTGRES_PASSWORD'], host='127.0.0.1',
    port=int(os.getenv('POSTGRES_PORT', '5432')),
    database=os.environ['POSTGRES_DB'],
)
engine = create_engine(url, pool_pre_ping=True, connect_args={'connect_timeout': 10})
try:
    with engine.connect() as conn:
        total = conn.execute(text('SELECT COUNT(*) FROM public.collisions')).scalar_one()
except Exception:
    raise RuntimeError(
        'Не удалось прочитать public.collisions. Проверьте Docker Desktop, '
        'docker compose ps, .env и создание таблицы sql/schema.sql.'
    ) from None
if total == 0:
    raise ValueError('Таблица пуста. Выполните python etl.py и дождитесь завершения.')
if etl_report and total != etl_report.get('rows_in_database'):
    raise ValueError('Число строк в БД отличается от последнего отчёта ETL.')
print(f'Подключение работает. Записей в БД: {total:,}')


In [ ]:
SOURCE = ROOT / 'data/raw/dft-road-casualty-statistics-collision-1979-latest-published-year.csv'
if SOURCE.is_file():
    source_bytes = SOURCE.stat().st_size
    print(f'Исходный файл: {source_bytes / 10**9:.2f} ГБ ({source_bytes / 1024**3:.2f} ГиБ)')
    if source_bytes <= 500_000_000:
        print('Проверьте требование задания: размер набора должен быть больше 0,5 ГБ.')
    if etl_report and source_bytes != etl_report.get('source_bytes'):
        raise ValueError('Размер текущего CSV отличается от файла последнего ETL.')
else:
    print('CSV локально отсутствует. Анализ использует БД; размер источника проверьте по отчёту ETL.')

with engine.connect() as conn:
    annual = pd.read_sql_query(text("""
        SELECT collision_year AS year, COUNT(*) AS collision_count
        FROM public.collisions GROUP BY collision_year ORDER BY collision_year
    """), conn)
    df = pd.read_sql_query(text("""
        SELECT collision_index, collision_year, collision_severity,
               speed_limit, light_conditions, date, time, police_force
        FROM public.collisions
        WHERE collision_year BETWEEN :start_year AND :end_year
    """), conn, params={'start_year': START_YEAR, 'end_year': END_YEAR})
if df.empty:
    raise ValueError('Нет данных за выбранный период.')
if int(annual['collision_count'].sum()) != total:
    raise ValueError('Годовая статистика не согласуется с общим количеством строк.')
annual.to_csv(RESULTS / 'annual.csv', index=False)
print(f'Записей за выбранный период: {len(df):,}')
display(df.head())
display(annual.tail(10))


## 6. Подготовка признаков и качество выборки

`date` и `time` уже типизированы в PostgreSQL. Признак `severe` равен 1 для кодов тяжести 1 и 2, равен 0 для кода 3. Неизвестная тяжесть остаётся пропуском. Ограничение скорости измеряется в mph. Код освещения 1 означает дневной свет, 6 — темноту без освещения.

Неизвестные значения исключаются из тех расчётов, которым соответствующий признак необходим. Код полицейского подразделения является категорией. Качество всей загрузки описано отчётом ETL, таблица ниже относится к выбранному периоду.


In [ ]:
if df['collision_index'].isna().any() or df['collision_index'].duplicated().any():
    raise ValueError('Нарушена уникальность или заполненность идентификаторов.')
for column in ['collision_year', 'collision_severity', 'speed_limit', 'light_conditions', 'police_force']:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].where(df['date'].dt.year.eq(df['collision_year']))
time_text = df['time'].astype('string')
valid_time = time_text.str.fullmatch(r'(?:[01]\d|2[0-3]):[0-5]\d(?::[0-5]\d(?:\.\d+)?)?').fillna(False)
df['hour'] = pd.to_numeric(time_text.where(valid_time).str[:2], errors='coerce')
df['weekday'] = df['date'].dt.dayofweek
df['severe'] = np.where(df['collision_severity'].isin([1, 2]), 1.0,
                        np.where(df['collision_severity'].eq(3), 0.0, np.nan))
quality = pd.DataFrame({
    'metric': ['rows_in_database', 'rows_in_period', 'unknown_severity',
               'unknown_or_other_speed', 'unknown_or_other_light', 'invalid_date_or_time'],
    'value': [total, len(df), int(df['severe'].isna().sum()),
              int((~df['speed_limit'].isin([20, 30, 40, 50, 60, 70])).sum()),
              int((~df['light_conditions'].isin([1, 4, 5, 6, 7])).sum()),
              int((df['date'].isna() | df['hour'].isna()).sum())],
})
quality.to_csv(RESULTS / 'quality.csv', index=False)
display(quality)


## 7. Гипотезы и статистические проверки

- **H1.** Среди зарегистрированных ДТП доля тяжёлых и смертельных на дорогах с лимитом 50–70 mph выше, чем на дорогах с лимитом 20–30 mph.
- **H2.** Доля тяжёлых и смертельных ДТП в темноте без уличного освещения выше, чем при дневном свете.
- **H3.** Среднее количество зарегистрированных ДТП в расчёте на час в интервалы 07:00–09:59 и 16:00–18:59 выше, чем в остальные часы.

Для каждой проверки нулевая гипотеза допускает равенство или отсутствие увеличения. Уровень значимости 0,05. Для H1 и H2 применяется односторонний Z-тест двух долей. Для H3 сравниваются парные дневные показатели с помощью одностороннего t-теста разностей. Поправка Бонферрони учитывает три проверки.

Пространственная и временная зависимость наблюдений может влиять на p-value. Вместе со значимостью необходимо оценивать величину различия. Эти проверки описывают ассоциации. Они не устанавливают причинный эффект скорости или освещения.


In [ ]:
def compare_shares(data, mask1, mask0, name, label1, label0):
    a = data.loc[mask1, 'severe'].dropna()
    b = data.loc[mask0, 'severe'].dropna()
    n1, n0 = len(a), len(b)
    x1, x0 = int(a.sum()), int(b.sum())
    if min(x1, n1 - x1, x0, n0 - x0) < 5:
        raise ValueError(f'{name}: недостаточно наблюдений для Z-теста.')
    p1, p0 = x1 / n1, x0 / n0
    pooled = (x1 + x0) / (n1 + n0)
    se = np.sqrt(pooled * (1 - pooled) * (1 / n1 + 1 / n0))
    z = (p1 - p0) / se
    table = pd.DataFrame({
        'group': [label1, label0], 'total': [n1, n0],
        'severe_count': [x1, x0], 'severe_percent': [100 * p1, 100 * p0],
    })
    result = {
        'hypothesis': name, 'group1': label1, 'group0': label0,
        'value1': 100 * p1, 'value0': 100 * p0,
        'difference': 100 * (p1 - p0), 'unit': 'percentage points',
        'statistic': float(z), 'test': 'one-sided two-proportion Z',
        'p_value': float(norm.sf(z)), 'log_p_value': float(norm.logsf(z)),
    }
    return table, result

speed_table, h1 = compare_shares(
    df, df['speed_limit'].isin([50, 60, 70]), df['speed_limit'].isin([20, 30]),
    'H1', '50–70 mph', '20–30 mph',
)
light_table, h2 = compare_shares(
    df, df['light_conditions'].eq(6), df['light_conditions'].eq(1),
    'H2', 'Без освещения', 'Дневной свет',
)
speed_table.to_csv(RESULTS / 'speed.csv', index=False)
light_table.to_csv(RESULTS / 'light.csv', index=False)
display(speed_table)
display(light_table)


In [ ]:
timed = df.dropna(subset=['date', 'hour']).copy()
timed['hour'] = timed['hour'].astype(int)
timed['peak'] = timed['hour'].isin([7, 8, 9, 16, 17, 18])
calendar = pd.date_range(f'{START_YEAR}-01-01', f'{END_YEAR}-12-31')
missing_days = calendar.difference(pd.DatetimeIndex(timed['date'].unique()))
if len(missing_days):
    raise ValueError(
        f'За {len(missing_days)} дней нет корректных наблюдений. '
        'Проверьте полноту периода. Отсутствующие данные нельзя автоматически считать нулём.'
    )
daily = timed.groupby(['date', 'peak']).size().unstack(fill_value=0)
daily = daily.reindex(index=calendar, columns=[True, False], fill_value=0)
daily.columns = ['peak_count', 'other_count']
daily.index.name = 'date'
daily['peak_per_hour'] = daily['peak_count'] / 6
daily['other_per_hour'] = daily['other_count'] / 18
daily['difference'] = daily['peak_per_hour'] - daily['other_per_hour']
difference = daily['difference']
if len(difference) < 30 or difference.std(ddof=1) == 0:
    raise ValueError('Недостаточно дней или вариации для проверки H3.')
test3 = ttest_1samp(difference, popmean=0, alternative='greater')
h3 = {
    'hypothesis': 'H3', 'group1': 'Часы пик', 'group0': 'Другие часы',
    'value1': float(daily['peak_per_hour'].mean()),
    'value0': float(daily['other_per_hour'].mean()),
    'difference': float(difference.mean()), 'unit': 'collisions/hour',
    'statistic': float(test3.statistic), 'test': 'one-sided paired daily t-test',
    'p_value': float(test3.pvalue),
}
daily.to_csv(RESULTS / 'daily.csv')
hourly = timed.groupby('hour').size().reindex(range(24), fill_value=0).rename('collision_count')
hourly.to_csv(RESULTS / 'hourly.csv')
heatmap = timed.groupby([timed['date'].dt.dayofweek.rename('weekday'), 'hour']).size().unstack(fill_value=0)
heatmap = heatmap.reindex(index=range(7), columns=range(24), fill_value=0)
heatmap.to_csv(RESULTS / 'heatmap.csv')
display(daily[['peak_per_hour', 'other_per_hour']].mean().to_frame('ДТП за час'))


In [ ]:
hypotheses = pd.DataFrame([h1, h2, h3])
hypotheses['p_adjusted'] = np.minimum(hypotheses['p_value'] * 3, 1)
hypotheses['decision'] = np.where(
    hypotheses['p_adjusted'] < ALPHA,
    'Отклоняем H0 в пользу направленной гипотезы',
    'Недостаточно оснований отклонить H0',
)
hypotheses.to_csv(RESULTS / 'hypotheses.csv', index=False)
display(hypotheses[['hypothesis', 'value1', 'value0', 'difference', 'unit', 'p_adjusted', 'decision']])
print('p-value, записанное как 0, означает численное округление очень малого значения.')


## 8. Корреляционный и регрессионный анализ

Для двух бинарных признаков коэффициент Пирсона совпадает с коэффициентом phi. Рассчитываем связь тяжести с группой высокого лимита и отсутствием освещения на сопоставимых подвыборках. Цифровые коды категорий освещения и полиции не используются как непрерывные величины.

Далее описательная логистическая регрессия учитывает группу лимита, освещение, год и полицейское подразделение. `exp(coefficient)` представляет отношение шансов при прочих включённых признаках. Модель регуляризована и не даёт статистических p-value для коэффициентов. Эти коэффициенты не интерпретируются как причинный эффект или изменение вероятности на столько же процентов.


In [ ]:
correlations = []
for feature, subset, values in [
    ('high_speed', df[df['speed_limit'].isin([20, 30, 50, 60, 70])].copy(), [50, 60, 70]),
    ('no_lighting', df[df['light_conditions'].isin([1, 6])].copy(), [6]),
]:
    subset = subset.dropna(subset=['severe'])
    source_column = 'speed_limit' if feature == 'high_speed' else 'light_conditions'
    binary = subset[source_column].isin(values).astype(int)
    coefficient = binary.corr(subset['severe']) if binary.nunique() > 1 and subset['severe'].nunique() > 1 else np.nan
    correlations.append({'feature': feature, 'n': len(subset), 'phi': coefficient})
correlation_table = pd.DataFrame(correlations)
correlation_table.to_csv(RESULTS / 'correlations.csv', index=False)
display(correlation_table)

model_data = df[
    df['speed_limit'].isin([20, 30, 50, 60, 70])
    & df['light_conditions'].isin([1, 6])
    & df['severe'].notna() & df['police_force'].gt(0)
].copy()
model_data['high_speed'] = model_data['speed_limit'].isin([50, 60, 70]).astype(int)
model_data['no_lighting'] = model_data['light_conditions'].eq(6).astype(int)
X_description = pd.get_dummies(
    model_data[['high_speed', 'no_lighting', 'collision_year', 'police_force']],
    columns=['collision_year', 'police_force'], drop_first=True, dtype=int,
)
y_description = model_data['severe'].astype(int)
if y_description.nunique() != 2:
    raise ValueError('Для регрессии нужны оба класса тяжести.')
descriptive_model = LogisticRegression(max_iter=2000)
descriptive_model.fit(X_description, y_description)
regression = pd.DataFrame({
    'feature': X_description.columns,
    'coefficient': descriptive_model.coef_[0],
    'odds_ratio': np.exp(descriptive_model.coef_[0]),
})
regression.to_csv(RESULTS / 'regression.csv', index=False)
print('Наблюдений в описательной регрессии:', len(model_data))
display(regression.head(2))


## 9. Машинное обучение: классификация тяжести ДТП

Дополнительная задача — предсказать тяжёлое/смертельное ДТП среди зарегистрированных ДТП. Обучение проводится на 2022–2023 годах, тестирование — на 2024 году. Это временное разделение. Тестовый набор не участвует в обучении преобразований.

Признаки — ограничение скорости, освещение, подразделение полиции, день недели и час. Категории кодируются One-Hot Encoder с обработкой новых значений. Пропуски заполняются отдельной категорией. Используется логистическая регрессия с балансировкой классов и порогом 0,5. Балансировка может влиять на калибровку вероятностей, поэтому их не следует считать готовой оценкой абсолютного риска.

Базовая модель всегда предсказывает наиболее частый класс обучения. Сравниваем accuracy, precision, recall, F1, ROC-AUC и average precision (AP). AP обобщает precision-recall-кривую, её базовый ориентир — доля положительного класса в тесте. Accuracy оценивается вместе с метриками редкого класса. Матрица ошибок показывает ложные тревоги и пропущенные тяжёлые ДТП. Порог и параметры по тесту не подбираются.


In [ ]:
FEATURES = ['speed_limit', 'light_conditions', 'police_force', 'weekday', 'hour']
predictive_data = df[df['severe'].notna()].copy()
X_predictive = predictive_data[FEATURES].copy()
for column in FEATURES:
    values = pd.to_numeric(X_predictive[column], errors='coerce')
    values = values.where(values.ge(0) & values.mod(1).eq(0))
    X_predictive[column] = values.astype('Int64').astype('string').fillna('unknown').astype(str)
y_predictive = predictive_data['severe'].astype(int)
train_mask = predictive_data['collision_year'].between(2022, 2023)
test_mask = predictive_data['collision_year'].eq(2024)
X_train, X_test = X_predictive.loc[train_mask], X_predictive.loc[test_mask]
y_train, y_test = y_predictive.loc[train_mask], y_predictive.loc[test_mask]
if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError('Обучающий и тестовый периоды должны содержать оба класса.')

encoder = ColumnTransformer([
    ('categories', OneHotEncoder(handle_unknown='ignore'), FEATURES),
])
classifier = Pipeline([
    ('features', encoder),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
])
classifier.fit(X_train, y_train)
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(np.zeros((len(y_train), 1)), y_train)

probabilities = classifier.predict_proba(X_test)[:, 1]
predictions = (probabilities >= 0.5).astype(int)
baseline_probabilities = baseline.predict_proba(np.zeros((len(y_test), 1)))[:, 1]
baseline_predictions = baseline.predict(np.zeros((len(y_test), 1)))

def score_model(name, actual, predicted, scores):
    return {
        'model': name, 'test_rows': len(actual),
        'accuracy': accuracy_score(actual, predicted),
        'precision': precision_score(actual, predicted, zero_division=0),
        'recall': recall_score(actual, predicted, zero_division=0),
        'f1': f1_score(actual, predicted, zero_division=0),
        'roc_auc': roc_auc_score(actual, scores),
        'average_precision': average_precision_score(actual, scores),
    }
ml_metrics = pd.DataFrame([
    score_model('Baseline: most frequent', y_test, baseline_predictions, baseline_probabilities),
    score_model('Logistic regression', y_test, predictions, probabilities),
])
ml_metrics.to_csv(RESULTS / 'ml_metrics.csv', index=False)
print(f'Обучение: {len(y_train):,}. Тест: {len(y_test):,}. Доля тяжёлых в тесте: {y_test.mean():.2%}')
display(ml_metrics)
cm = confusion_matrix(y_test, predictions, labels=[0, 1])
pd.DataFrame(cm, index=['actual_light', 'actual_severe'],
             columns=['predicted_light', 'predicted_severe']).to_csv(RESULTS / 'confusion_matrix.csv')
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay(cm, display_labels=['Лёгкое', 'Тяжёлое']).plot(ax=axes[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, probabilities, ax=axes[1], name='Logistic regression')
PrecisionRecallDisplay.from_predictions(y_test, probabilities, ax=axes[2], name='Logistic regression')
axes[2].axhline(y_test.mean(), linestyle='--', color='gray', label='Доля положительного класса')
axes[2].legend()
plt.tight_layout()
plt.savefig(RESULTS / 'ml_evaluation.png', dpi=150)
plt.show()
plt.close(fig)


## 10. Интерактивная визуализация

Графики Plotly поддерживают подсказки, масштабирование и сохранение изображения. Сравнения по скорости и освещению имеют переключатель года. Каждый HTML-файл содержит библиотеку Plotly и открывается отдельно в браузере без Python. Если VS Code не показывает интерактивный вывод, откройте HTML из `results` через Finder. Для показа в самом ноутбуке может потребоваться доверие к файлу и доступ к CDN.

Показатели тяжести рассчитываются среди записей с известной тяжестью в соответствующей группе. Переключатель года меняет числитель и знаменатель. Временные графики отражают число зарегистрированных ДТП. Данные о числе поездок и интенсивности движения отсутствуют.


In [ ]:
def export_interactive(fig, filename):
    fig.write_html(RESULTS / filename, include_plotlyjs=True, full_html=True)
    display(fig)

fig_annual = px.line(annual, x='year', y='collision_count', markers=True,
                    title='Количество зарегистрированных ДТП по годам',
                    labels={'year': 'Год', 'collision_count': 'Количество ДТП'})
export_interactive(fig_annual, 'annual_interactive.html')

fig_hourly = px.bar(hourly.reset_index(), x='hour', y='collision_count',
                   title=f'Распределение ДТП по часам, {START_YEAR}–{END_YEAR}',
                   labels={'hour': 'Час', 'collision_count': 'Количество ДТП'})
export_interactive(fig_hourly, 'hourly_interactive.html')

fig_heatmap = go.Figure(go.Heatmap(
    z=heatmap.values, x=list(range(24)),
    y=['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс'], colorscale='Blues',
    colorbar={'title': 'ДТП'},
))
fig_heatmap.update_layout(title='ДТП по дням недели и часам', xaxis_title='Час')
export_interactive(fig_heatmap, 'heatmap_interactive.html')

def yearly_share_chart(kind):
    is_speed = kind == 'speed'
    labels = ['50–70 mph', '20–30 mph'] if is_speed else ['Без освещения', 'Дневной свет']
    fig = go.Figure()
    years = ['Весь период', *sorted(df['collision_year'].dropna().astype(int).unique())]
    exports = []
    for i, year in enumerate(years):
        subset = df if year == 'Весь период' else df[df['collision_year'].eq(year)]
        masks = ([subset['speed_limit'].isin([50, 60, 70]), subset['speed_limit'].isin([20, 30])]
                 if is_speed else [subset['light_conditions'].eq(6), subset['light_conditions'].eq(1)])
        values, details = [], []
        for label, mask in zip(labels, masks):
            known = subset.loc[mask, 'severe'].dropna()
            share = float(known.mean() * 100) if len(known) else None
            values.append(share)
            details.append([len(known), int(known.sum())])
            exports.append({'period': str(year), 'group': label, 'total': len(known),
                            'severe_count': int(known.sum()), 'severe_percent': share})
        fig.add_trace(go.Bar(
            x=labels, y=values, customdata=details, visible=(i == 0), name=str(year),
            hovertemplate='%{x}<br>Доля: %{y:.2f}%<br>Всего: %{customdata[0]}<br>Тяжёлых: %{customdata[1]}<extra></extra>',
        ))
    buttons = [dict(label=str(year), method='update',
                    args=[{'visible': [j == i for j in range(len(years))]}])
               for i, year in enumerate(years)]
    fig.update_layout(
        title='Доля тяжёлых и смертельных ДТП ' + ('по ограничению скорости' if is_speed else 'по освещению'),
        yaxis_title='Доля, %', updatemenus=[{'buttons': buttons, 'x': 1.0, 'y': 1.2}],
        yaxis_range=[0, 100], showlegend=False,
    )
    pd.DataFrame(exports).to_csv(RESULTS / f'{kind}_by_year.csv', index=False)
    return fig

export_interactive(yearly_share_chart('speed'), 'speed_interactive.html')
export_interactive(yearly_share_chart('light'), 'light_interactive.html')
print('HTML-графики сохранены в', RESULTS)


In [ ]:
# Компактные статические графики для отчёта.
for table, name, title in [
    (speed_table, 'speed.png', 'Доля тяжёлых ДТП по ограничению скорости'),
    (light_table, 'light.png', 'Доля тяжёлых ДТП по освещению'),
]:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(table['group'], table['severe_percent'])
    ax.set(ylabel='Доля, %', title=title)
    fig.tight_layout()
    fig.savefig(RESULTS / name, dpi=150)
    plt.close(fig)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(annual['year'], annual['collision_count'])
ax.set(xlabel='Год', ylabel='Количество ДТП', title='Годовая динамика')
fig.tight_layout()
fig.savefig(RESULTS / 'annual.png', dpi=150)
plt.close(fig)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(hourly.index, hourly.values)
ax.set(xlabel='Час', ylabel='Количество ДТП', title='ДТП по часам', xticks=range(24))
fig.tight_layout()
fig.savefig(RESULTS / 'hourly.png', dpi=150)
plt.close(fig)
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(heatmap.values, aspect='auto', cmap='Blues')
ax.set(xticks=range(24), yticks=range(7), yticklabels=['Пн','Вт','Ср','Чт','Пт','Сб','Вс'], xlabel='Час')
fig.colorbar(im, ax=ax, label='Количество ДТП')
fig.tight_layout()
fig.savefig(RESULTS / 'heatmap.png', dpi=150)
plt.close(fig)


## 11. Superset и дашборд

### Запуск интерфейса

Для уже скачанной официальной конфигурации версии 6.0.0 в папке `~/Projects/superset-tools`:

```bash
cd ~/Projects/superset-tools
TAG=6.0.0-dev docker compose -f docker-compose-image-tag.yml up -d
TAG=6.0.0-dev docker compose -f docker-compose-image-tag.yml ps -a
```

Перед первым запуском в `docker/.env-local` должен быть задан `SUPERSET_SECRET_KEY` со случайным значением, полученным командой `openssl rand -hex 42`. Для отключения демонстрационных данных можно добавить `SUPERSET_LOAD_EXAMPLES=no`. Дождитесь успешного завершения `superset-init`. Откройте `http://localhost:8088`. В стандартной учебной конфигурации начальная учётная запись — `admin` / `admin`.

Если папка ещё не скачана, сначала выполните в терминале:

```bash
cd ~/Projects
git clone --depth 1 --branch 6.0.0 https://github.com/apache/superset.git superset-tools
```

### Подключение данных

Откройте **Settings → Database Connections → + Database → PostgreSQL**. Название подключения — `Road Safety`. Для отдельного Compose на Mac задайте Host `host.docker.internal`, Port из `.env` проекта, Database `road_safety`, Username `road_user` и фактический пароль из `.env`. Нажмите **Test Connection**, затем сохраните подключение. При другой сетевой конфигурации используйте соответствующее имя сервиса.

Для вычисляемых полей создадим представление `public.collisions_dashboard`. Оно предоставляет год, дату, час, день недели, тяжесть и читаемые категории. Представление не создаёт вторую копию всех строк. Следующая ячейка выполняет `CREATE OR REPLACE VIEW` в вашей БД.


In [ ]:
view_sql = """
CREATE OR REPLACE VIEW public.collisions_dashboard AS
SELECT
    collision_index, collision_year, date AS collision_date,
    EXTRACT(HOUR FROM time)::integer AS hour,
    EXTRACT(ISODOW FROM date)::integer AS weekday,
    speed_limit, light_conditions, police_force, collision_severity,
    CASE WHEN collision_severity IN (1, 2) THEN 1
         WHEN collision_severity = 3 THEN 0 END AS severe,
    CASE WHEN speed_limit IN (50, 60, 70) THEN '50–70 mph'
         WHEN speed_limit IN (20, 30) THEN '20–30 mph'
         WHEN speed_limit = 40 THEN '40 mph'
         ELSE 'Неизвестно/другое' END AS speed_group,
    CASE WHEN light_conditions = 1 THEN 'Дневной свет'
         WHEN light_conditions = 6 THEN 'Без освещения'
         ELSE 'Другие/неизвестные условия' END AS lighting_group
FROM public.collisions
"""
with engine.begin() as conn:
    conn.execute(text(view_sql))
print('Представление public.collisions_dashboard готово для Superset.')


### Сборка дашборда

1. Откройте **Data → Datasets → + Dataset**. Выберите подключение `Road Safety`, схему `public` и представление `collisions_dashboard`. При необходимости включите показ views в выборе объектов. Добавьте датасет.
2. В настройках датасета отметьте `collision_date` как временное поле. Для целочисленных категорий `collision_year`, `hour`, `weekday`, `speed_group`, `lighting_group` разрешите группировки и фильтры.
3. Создайте показатель доли тяжёлых ДТП с SQL-выражением `100.0 * SUM(severe) / NULLIF(COUNT(severe), 0)`. Знаменатель учитывает только известную тяжесть. Показатель количества — `COUNT(*)`.
4. Через **Charts → + Chart** создайте графики из таблицы ниже и сохраните их в общий дашборд «Анализ ДТП».
5. В режиме редактирования дашборда добавьте фильтр года. Для исследовательских графиков по умолчанию выберите 2022–2024. Историческому графику оставьте весь период через настройку области действия фильтра. Добавьте фильтры ограничения скорости и освещения.
6. Проверьте, что изменение фильтров действительно меняет показатели. Сохраните дашборд и экспортируйте его через меню Superset в ZIP для передачи команде.

| График | Группировка | Показатель и фильтры |
|---|---|---|
| Годовая динамика | `collision_year` | `COUNT(*)`, все годы |
| Доля тяжёлых по скорости | `speed_group` | Показатель доли, группы 20–30 и 50–70 mph |
| Доля тяжёлых по освещению | `lighting_group` | Показатель доли, дневной свет и без освещения |
| Распределение по часам | `hour` | `COUNT(*)`, исключить пустой час |
| Тепловая карта | `weekday` и `hour` | `COUNT(*)`, исключить пустые значения |

При сравнении с ноутбуком задавайте те же годы и условия. Фильтруемые графики Superset не пересчитывают статистические тесты и ML-модель из Python автоматически.


## 12. Выводы по фактическому запуску

Следующая ячейка формирует выводы из рассчитанных таблиц и сохраняет их в `results/conclusions.md`. Ограничения сохраняются даже при малых p-value и высоких метриках.


In [ ]:
lines = [
    '# Результаты анализа ДТП', '',
    f'В PostgreSQL {total:,} записей. В периоде {START_YEAR}–{END_YEAR} — {len(df):,}.', '',
]
for row in hypotheses.itertuples():
    unit = 'процентных пункта' if row.unit == 'percentage points' else 'ДТП в час'
    p_text = '< 1e-300 (численное округление)' if row.p_adjusted == 0 else f'{row.p_adjusted:.3g}'
    lines.append(
        f'- {row.hypothesis}: {row.group1} — {row.value1:.2f}, '
        f'{row.group0} — {row.value0:.2f}. Разница {row.difference:.2f} {unit}. '
        f'Скорректированное p-value {p_text}. {row.decision}.'
    )
lines += ['', '## Классификация на 2024 год', '']
for row in ml_metrics.itertuples():
    lines.append(
        f'- {row.model}: accuracy {row.accuracy:.3f}, precision {row.precision:.3f}, '
        f'recall {row.recall:.3f}, F1 {row.f1:.3f}, '
        f'ROC-AUC {row.roc_auc:.3f}, AP {row.average_precision:.3f}.'
    )
lines += [
    '', '## Ограничения', '',
    '- STATS19 содержит зарегистрированные полицией ДТП с пострадавшими. Изменения регистрации и полноты данных ограничивают сравнение лет.',
    '- Ограничение скорости отличается от фактической скорости автомобиля.',
    '- Без данных о транспортном потоке число ДТП нельзя интерпретировать как риск на поездку.',
    '- Статистические тесты предполагают условия независимости. Временная и пространственная зависимость могут влиять на значимость.',
    '- Наблюдаемые связи не устанавливают причинность. В модели отсутствуют многие характеристики дороги, автомобиля и участника.',
    '- Классификация оценивает тяжесть среди уже зарегистрированных ДТП. Предсказание самого возникновения ДТП не исследуется.',
    '- Метрики теста характеризуют период 2024 года. Балансированная модель требует отдельной калибровки перед интерпретацией вероятностей как абсолютных оценок.',
]
conclusions = '\n'.join(lines)
(RESULTS / 'conclusions.md').write_text(conclusions, encoding='utf-8')
display(Markdown(conclusions))
print('Результаты сохранены в', RESULTS)
engine.dispose()


## 13. Проверка требований задания и источники

| Требование | Где проверяется |
|---|---|
| Открытый набор больше 0,5 ГБ | Размер CSV и отчёт ETL |
| Архитектура хранения и обработки | Разделы 1–3, PostgreSQL и Docker |
| Очистка и преобразования | `etl.py`, отчёт ETL и контроль качества |
| Методы интеллектуального анализа | Корреляционный и регрессионный анализ |
| Три проверенные гипотезы | H1–H3 и `hypotheses.csv` |
| Интерактивные визуализации | HTML-графики Plotly и отдельно настраиваемый дашборд Superset |
| Дополнительная классификация | Временное разделение обучения и теста, `ml_metrics.csv` |
| Дополнительные внешние данные | В текущей реализации не используются |

Завершённость ETL подтверждается отчётом и количеством строк в БД. Завершённость Superset подтверждается фактически сохранённым дашбордом, работой фильтров и его экспортом. Эти действия выполняются участниками после запуска.

Источники и документация:

- [STATS19 — Road safety open data, Department for Transport](https://www.gov.uk/government/statistical-data-sets/road-safety-open-data). Зафиксируйте точную ссылку и дату скачивания вашего CSV в README. Состав кодов сверяйте со словарём соответствующей версии.
- [PostgreSQL Docker Official Image](https://hub.docker.com/_/postgres).
- [Docker Compose](https://docs.docker.com/compose/).
- [Psycopg COPY](https://www.psycopg.org/psycopg3/docs/basic/copy.html).
- [Superset Quickstart](https://superset.apache.org/user-docs/6.0.0/quickstart/).
- [Scikit-learn — LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

В репозиторий добавляются код, ноутбук, SQL, Compose, инструкция и экспорт дашборда. Исходный CSV и `.env` остаются локально. Фактические версии Python-пакетов сохраняются при запуске в `results/analysis_environment.json`.
